# Clone your voice for the Chokepoint Ledger videos

Free GPU narration with **Chatterbox** (open source, MIT licence). About 15 minutes end to end.

1. **Runtime > Change runtime type > T4 GPU**, then run the cells in order (Shift+Enter).
2. Record or upload about 60 s of your voice reading the reference passage.
3. Preview a line, tweak the settings if needed, then generate every scene.
4. Download `narration_<episode>.zip` and unzip it into `video/recordings/` in the repo (or send it to Claude to render).

Your reference audio stays in this Colab session and is deleted when the session ends. Don't share it, because it is enough to clone your voice.

In [ ]:
#@title 1. Install Chatterbox (takes 2 to 3 minutes)
import torch, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none. Switch the runtime to T4 GPU first")
!pip -q install chatterbox-tts pyyaml soundfile
print("Installed. If Colab asks you to restart the session, do it, then carry on from step 2.")

## 2. Your reference recording
Use **2a** to record in the browser or **2b** to upload a file you recorded on your phone or with `voice_kit.py record --reference`. Quiet room, normal speaking pace, mic 15 to 25 cm away.

In [ ]:
#@title 2a. Record in the browser (click Start, read the passage, click Stop)
from IPython.display import display, Javascript, HTML
from google.colab import output
import base64

PASSAGE = "I study engineering, and this channel is about the physical side of the economy.  Where crude oil, diesel, fertiliser, grain and metals actually move, and what happens when a route closes.  Take sulphur. Most of it is a by-product of cleaning sour gas, so producers cannot simply make more when the price rises.  Is that a problem? It is if you run a copper leach plant in Chile, or a nickel refinery in Indonesia.  Numbers help. Ten million barrels a day, fourteen thousand eight hundred dollars a tonne, forty five percent.  Honestly, the most interesting part is how one input links energy, metals and food.  So let's map it, score it, and see which supply lines break first."
display(HTML(f"<div style='font:20px/1.5 system-ui;max-width:46ch;padding:12px;border-left:4px solid #5cb2d4'>{PASSAGE}</div>"))
RECORD_JS = '''
async function rec(){
  const box=document.createElement('div');box.style='margin:12px 0;font:16px system-ui';
  const b=document.createElement('button');b.textContent='Start recording';b.style='font:16px system-ui;padding:10px 16px';
  const s=document.createElement('span');s.style='margin-left:12px';box.append(b,s);document.body.append(box);
  await new Promise(r=>b.onclick=r);
  const st=await navigator.mediaDevices.getUserMedia({audio:{echoCancellation:false,noiseSuppression:false,autoGainControl:false}});
  const mr=new MediaRecorder(st);const ch=[];mr.ondataavailable=e=>ch.push(e.data);mr.start();
  const t0=Date.now();const iv=setInterval(()=>s.textContent='Recording '+((Date.now()-t0)/1000).toFixed(0)+' s',250);
  b.textContent='Stop';await new Promise(r=>b.onclick=r);
  await new Promise(r=>{mr.onstop=r;mr.stop()});clearInterval(iv);st.getTracks().forEach(t=>t.stop());
  s.textContent='Saved';const buf=await new Blob(ch).arrayBuffer();
  let bin='';const u=new Uint8Array(buf);for(let i=0;i<u.length;i++)bin+=String.fromCharCode(u[i]);return btoa(bin);
}
rec();'''
data = output.eval_js(RECORD_JS, timeout_sec=600)
open("reference_raw.webm", "wb").write(base64.b64decode(data))
print("Recorded reference_raw.webm. Now run step 2c.")

In [ ]:
#@title 2b. Or upload a recording (wav, mp3, m4a, webm)
from google.colab import files
up = files.upload()
name = next(iter(up))
import shutil; shutil.move(name, "reference_raw" + name[name.rfind("."):])
print("Uploaded", name, "Now run step 2c.")

In [ ]:
#@title 2c. Clean the recording and keep the best 20 s
import glob, subprocess
from IPython.display import Audio, display
raw = sorted(glob.glob("reference_raw.*"))[-1]
subprocess.run(["ffmpeg","-y","-v","error","-i",raw,"-ac","1","-ar","24000","-af",
  "highpass=f=70,afftdn=nf=-25,silenceremove=start_periods=1:start_threshold=-45dB:stop_periods=-1:stop_threshold=-45dB:stop_duration=0.6,loudnorm=I=-18:TP=-2",
  "reference_clean.wav"], check=True)
total = float(subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0","reference_clean.wav"],capture_output=True,text=True).stdout)
length = min(20.0, total); start = max(0.0, min(2.0, total-length))
subprocess.run(["ffmpeg","-y","-v","error","-ss",str(start),"-t",str(length),"-i","reference_clean.wav","reference.wav"], check=True)
print(f"reference.wav: {length:.0f} s of {total:.0f} s speech" + ("  (under 12 s, re-record for a better clone)" if total < 12 else ""))
display(Audio("reference.wav"))

In [ ]:
%%writefile clone_narration.py
#!/usr/bin/env python3
"""
Generate an episode's narration in YOUR cloned voice with Chatterbox (open source, MIT licence).

Runs anywhere Chatterbox runs: Google Colab (free GPU, see clone_voice_colab.ipynb), a PC with an
NVIDIA GPU, an Apple Silicon Mac (MPS), or plain CPU (slow). Output is one wav per scene plus a
timing file, named so make_video.py can use it directly:

  python video/clone_narration.py video/episodes/ep01_chokepoints.yaml --ref video/my_voice/reference.wav
  python video/make_video.py video/episodes/ep01_chokepoints.yaml --tts own

Tuning (try on one line first with --preview "some text"):
  --exaggeration  0.3 calm and flat ... 0.5 default ... 0.7 animated
  --cfg           0.5 default. Lower (0.3) if the clone talks too fast or sounds rushed
  --temperature   0.6 to 0.8. Lower is steadier, higher is more varied
  --seed          keep fixed so the voice stays consistent between scenes and episodes
"""
import argparse, json, re, sys, wave
from pathlib import Path

import yaml

GAP = 0.28  # seconds of silence between sentences, matches make_video.py

# Expand units and acronyms so the voice reads them naturally (captions keep the written form)
SPOKEN = [
    (r"\bmb/d\b", "million barrels a day"), (r"\bkb/d\b", "thousand barrels a day"), (r"\bMtpa\b", "million tonnes a year"),
    (r"\bMt\b", "million tonnes"), (r"\$/bbl", "dollars a barrel"), (r"\$/t\b", "dollars a tonne"),
    (r"\$([\d,\.]+)\s*/bbl", r"\1 dollars a barrel"), (r"\$([\d,\.]+)\s*/t\b", r"\1 dollars a tonne"),
    (r"\$([\d,\.]+)", r"\1 dollars"), (r"(\d)%", r"\1 percent"), (r"\bSX-EW\b", "S X E W"), (r"\bHPAL\b", "H-pal"),
    (r"\bLNG\b", "L N G"), (r"\bDAP\b", "D A P"), (r"\bIEA\b", "I E A"), (r"\bFAO\b", "F A O"), (r"\bOCP\b", "O C P"),
    (r"\bShFE\b", "Shanghai Futures Exchange"), (r"\bLME\b", "L M E"), (r"\bREE?s?\b", "rare earths"), (r"\be\.g\.", "for example"),
    (r"\bvs\b", "versus"), (r"\bDy\b", "dysprosium"), (r"\bTb\b", "terbium"),
]


def spoken(t):
    for a, b in SPOKEN:
        t = re.sub(a, b, t)
    return t


def sentences(text):
    text = " ".join((text or "").split())
    return [p for p in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9$\"'])", text) if p.strip()]


def episode_lines(ep):
    """(scene id, narration) for every voiced scene, including Shorts hooks and outros."""
    out = [(s["id"], s.get("narration", "")) for s in ep["scenes"] if s.get("narration")]
    for sh in ep.get("shorts", []):
        if sh.get("hook", {}).get("say"):
            out.append((f"{sh['id']}_hook", sh["hook"]["say"]))
        if sh.get("outro"):
            out.append((f"{sh['id']}_outro", sh["outro"]))
    return out


class Cloner:
    def __init__(self, ref, exaggeration, cfg, temperature, seed):
        import torch
        from chatterbox.tts import ChatterboxTTS
        self.torch = torch
        dev = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
        print(f"Loading Chatterbox on {dev} (first run downloads about 3 GB)")
        self.m = ChatterboxTTS.from_pretrained(device=dev)
        self.m.prepare_conditionals(str(ref), exaggeration=exaggeration)
        self.sr = self.m.sr
        self.cfg, self.temperature, self.seed = cfg, temperature, seed

    def say(self, text):
        if self.seed is not None:
            self.torch.manual_seed(self.seed)
        wav = self.m.generate(spoken(text), cfg_weight=self.cfg, temperature=self.temperature)
        return wav.squeeze(0).detach().cpu().numpy()


def write_wav(path, chunks, sr):
    import numpy as np
    gap = np.zeros(int(GAP * sr), dtype=np.float32)
    timings, parts, t = [], [], 0.0
    for text, audio in chunks:
        audio = audio.astype(np.float32)
        d = len(audio) / sr
        timings.append([text, round(t, 3), round(t + d, 3)])
        parts += [audio, gap]
        t += d + GAP
    data = np.concatenate(parts) if parts else gap
    peak = float(np.abs(data).max()) or 1.0
    pcm = (data / max(peak, 1e-6) * 0.89 * 32767).astype("<i2")  # normalise to -1 dBFS peak
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr); w.writeframes(pcm.tobytes())
    return timings


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("episodes", nargs="*", help="episode yaml files")
    ap.add_argument("--ref", default=str(Path(__file__).resolve().parent / "my_voice" / "reference.wav"))
    ap.add_argument("--out", default=str(Path(__file__).resolve().parent / "recordings"))
    ap.add_argument("--scenes", help="comma list of scene ids to (re)generate")
    ap.add_argument("--preview", help="just speak this text to preview.wav")
    ap.add_argument("--exaggeration", type=float, default=0.5)
    ap.add_argument("--cfg", type=float, default=0.5)
    ap.add_argument("--temperature", type=float, default=0.7)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--force", action="store_true", help="overwrite scenes that already exist")
    a = ap.parse_args()

    if not Path(a.ref).exists():
        sys.exit(f"No reference recording at {a.ref}")
    c = Cloner(a.ref, a.exaggeration, a.cfg, a.temperature, a.seed)

    if a.preview:
        write_wav(Path("preview.wav"), [(s, c.say(s)) for s in sentences(a.preview)], c.sr)
        print("Wrote preview.wav")
        return

    sel = set(a.scenes.split(",")) if a.scenes else None
    for ep_path in a.episodes:
        ep = yaml.safe_load(Path(ep_path).read_text())
        d = Path(a.out) / ep["id"]
        d.mkdir(parents=True, exist_ok=True)
        lines = [(sid, txt) for sid, txt in episode_lines(ep) if not sel or sid in sel]
        print(f"{ep['title']}: {len(lines)} scenes -> {d}")
        for n, (sid, txt) in enumerate(lines, 1):
            wav_path = d / f"{sid}.wav"
            if wav_path.exists() and not a.force and not sel:
                print(f"  [{n}/{len(lines)}] {sid} exists, skipping")
                continue
            sents = sentences(txt)
            chunks = []
            for s in sents:
                chunks.append((s, c.say(s)))
            timings = write_wav(wav_path, chunks, c.sr)
            (d / f"{sid}.json").write_text(json.dumps({"engine": "chatterbox-clone", "sentences": timings}, indent=1))
            print(f"  [{n}/{len(lines)}] {sid}: {timings[-1][2]:.1f} s")
    print("Done. Render with: python video/make_video.py <episode.yaml> --tts own")


if __name__ == "__main__":
    main()


## 3. Episode scripts
Upload the episode files from `video/episodes/` (you can select several), or set `REPO` to your GitHub repo once it is public.

In [ ]:
#@title 3. Get the episode YAML files
REPO = ""  #@param {type:"string"}
EPISODES = "ep01_chokepoints.yaml,ep02_spot_vs_producers.yaml"  #@param {type:"string"}
import os, urllib.request
os.makedirs("episodes", exist_ok=True)
if REPO:
    for e in EPISODES.split(","):
        url = f"https://raw.githubusercontent.com/{REPO}/main/video/episodes/{e.strip()}"
        urllib.request.urlretrieve(url, f"episodes/{e.strip()}"); print("fetched", e)
else:
    from google.colab import files
    for n, b in files.upload().items():
        open(f"episodes/{n}", "wb").write(b); print("uploaded", n)

In [ ]:
#@title 4. Preview your cloned voice on one line
TEXT = "Sulphur is the commodity almost nobody is watching. Because it is a by-product, no producer can increase output when the price goes up."  #@param {type:"string"}
EXAGGERATION = 0.5  #@param {type:"slider", min:0.25, max:0.9, step:0.05}
CFG = 0.5  #@param {type:"slider", min:0.2, max:0.8, step:0.05}
TEMPERATURE = 0.7  #@param {type:"slider", min:0.5, max:0.9, step:0.05}
SEED = 7  #@param {type:"integer"}
!python clone_narration.py --ref reference.wav --preview "$TEXT" --exaggeration $EXAGGERATION --cfg $CFG --temperature $TEMPERATURE --seed $SEED
from IPython.display import Audio, display
display(Audio("preview.wav"))
print("Sounds rushed: lower CFG to about 0.3. Too flat: raise EXAGGERATION. Unstable: lower TEMPERATURE or try another SEED.")

In [ ]:
#@title 5. Generate every scene and download the zip
import glob, shutil, os
eps = " ".join(sorted(glob.glob("episodes/*.yaml")))
!python clone_narration.py $eps --ref reference.wav --out recordings --exaggeration $EXAGGERATION --cfg $CFG --temperature $TEMPERATURE --seed $SEED
from google.colab import files
for d in sorted(glob.glob("recordings/*")):
    z = shutil.make_archive(f"narration_{os.path.basename(d)}", "zip", d)
    files.download(z)
print("Unzip each file into video/recordings/<episode id>/ and run: python video/make_video.py <episode.yaml> --tts own")

In [ ]:
#@title 6. (Optional) Re-do specific scenes
SCENES = "sulphur,ranking"  #@param {type:"string"}
!python clone_narration.py $eps --ref reference.wav --out recordings --scenes "$SCENES" --exaggeration $EXAGGERATION --cfg $CFG --temperature $TEMPERATURE --seed $SEED
from IPython.display import Audio, display
for s in SCENES.split(","):
    for f in glob.glob(f"recordings/*/{s.strip()}.wav"): print(f); display(Audio(f))